# ☀️ Sunrise Super Procurement Assistant — ML Pipeline
### Project: 2026-Y2-S2-KU-AI-02

This notebook replaces the rule-based Node.js analysis with a full **AI/ML pipeline** consisting of:
1. 📥 **Data Fetching** from Supabase
2. 🔧 **Feature Engineering** (velocity, coverage, demand variability)
3. 🔵 **K-Means Clustering** — segment SKUs (Fast / Medium / Slow / Dead)
4. 🟠 **XGBoost Regression** — predict suggested reorder quantity
5. 🟢 **Random Forest Classification** — predict urgency tier
6. 🔮 **Prophet Forecasting** — time-series demand forecast (for SKUs with enough data)
7. 📤 **Upload to Supabase** — push ML predictions to `replenishment_orders` table

---
The **Procurement Dashboard** at `http://localhost:3456` reads directly from `replenishment_orders` — no code changes needed there.

## Step 0 — Install Dependencies

In [ ]:
!pip install -q pandas numpy scikit-learn xgboost prophet requests matplotlib seaborn joblib

## Step 1 — Configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import requests, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from datetime import datetime, timezone
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, mean_absolute_error
import xgboost as xgb

matplotlib.rcParams['figure.facecolor'] = '#0a0d14'
matplotlib.rcParams['axes.facecolor']   = '#111827'
matplotlib.rcParams['text.color']       = '#f1f5f9'
matplotlib.rcParams['axes.labelcolor']  = '#94a3b8'
matplotlib.rcParams['xtick.color']      = '#475569'
matplotlib.rcParams['ytick.color']      = '#475569'
matplotlib.rcParams['axes.edgecolor']   = '#1a2235'

ACCENT   = '#f59e0b'
GREEN    = '#10b981'
RED      = '#ef4444'
BLUE     = '#3b82f6'
PURPLE   = '#8b5cf6'

# ── Supabase credentials ──────────────────────────────────────────────────────
SUPABASE_URL = 'https://obbctmyetrdortxyglle.supabase.co'
SUPABASE_KEY = ('eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJpc3MiOiJzdXBhYmFzZSIsInJlZiI6Im9iYmN0bXlldHJkb3J0'
                'eHlnbGxlIiwicm9sZSI6InNlcnZpY2Vfcm9sZSIsImlhdCI6MTc3MTY1OTg1MywiZXhwIjoyMDg3MjM1ODUzfQ.'
                'uaOYLnsgpN7R4mcbd4qcd8_jY4swh18Cf8sKF063LOg')

HEADERS = {
    'apikey':        SUPABASE_KEY,
    'Authorization': f'Bearer {SUPABASE_KEY}',
    'Content-Type':  'application/json',
    'Prefer':        'return=representation',
}

# ── ML Hyperparameters ────────────────────────────────────────────────────────
N_CLUSTERS       = 4
LEAD_TIME_DAYS   = 7
COVERAGE_DAYS    = 30
SAFETY_BUFFER    = 0.10
MIN_PROPHET_DAYS = 14

print('✅ Configuration loaded')


## Step 2 — Fetch Data from Supabase

In [ ]:
def fetch_table(name, limit=10000):
    PAGE = 1000
    rows = []
    offset = 0
    enc = requests.utils.quote(name)
    while True:
        url = f'{SUPABASE_URL}/rest/v1/{enc}?select=*&limit={PAGE}&offset={offset}'
        r = requests.get(url, headers=HEADERS)
        data = r.json()
        rows.extend(data)
        if len(data) < PAGE: break
        offset += PAGE
        if offset >= limit: break
    return rows

print('📥 Fetching tables...')
sales_rows    = fetch_table('His Sales')
po_rows       = fetch_table('PO')
item_rows     = fetch_table('Item Master')
wastage_rows  = fetch_table('Wastage')

print(f'   His Sales:    {len(sales_rows):,} rows')
print(f'   PO:           {len(po_rows):,} rows')
print(f'   Item Master:  {len(item_rows):,} rows')
print(f'   Wastage:      {len(wastage_rows):,} rows')


## Step 3 — Feature Engineering

In [ ]:
# ── Item lookup ───────────────────────────────────────────────────────────────
item_df = pd.DataFrame(item_rows)
item_lookup = {}
if not item_df.empty and 'Prod_Code' in item_df.columns:
    for _, r in item_df.iterrows():
        item_lookup[str(r['Prod_Code']).strip()] = {
            'name':       str(r.get('Prod_Name', 'UNKNOWN')).strip(),
            'cost_price': float(r.get('Prod_StadCostPrice', 0) or 0),
        }

# ── Sales features ────────────────────────────────────────────────────────────
sales_df = pd.DataFrame(sales_rows)
sales_df['Quantity']    = pd.to_numeric(sales_df['Quantity'],   errors='coerce').fillna(0)
sales_df['Unit_Price']  = pd.to_numeric(sales_df['Unit_Price'], errors='coerce').fillna(0)
sales_df['date_parsed'] = pd.to_datetime(sales_df['Date'].astype(str), format='%Y%m%d', errors='coerce')
sales_df = sales_df.dropna(subset=['date_parsed'])

date_range_days = max((sales_df['date_parsed'].max() - sales_df['date_parsed'].min()).days + 1, 1)
print(f'📅 Sales date range: {sales_df["date_parsed"].min().date()} → {sales_df["date_parsed"].max().date()} ({date_range_days} days)')

daily = sales_df.groupby(['Product_Code', 'date_parsed'])['Quantity'].sum().reset_index()
agg = daily.groupby('Product_Code').agg(
    total_qty_sold  = ('Quantity', 'sum'),
    days_with_sales = ('date_parsed', 'nunique'),
    avg_daily_qty   = ('Quantity', 'mean'),
    std_daily_qty   = ('Quantity', 'std'),
).reset_index()
agg['std_daily_qty']      = agg['std_daily_qty'].fillna(0)
agg['cv_demand']          = np.where(agg['avg_daily_qty'] > 0, agg['std_daily_qty'] / agg['avg_daily_qty'], 0)
agg['avg_daily_qty_norm'] = agg['total_qty_sold'] / date_range_days

sales_df['revenue'] = sales_df['Quantity'] * sales_df['Unit_Price']
rev = sales_df.groupby('Product_Code').agg(total_revenue=('revenue','sum'), avg_unit_price=('Unit_Price','mean')).reset_index()
agg = agg.merge(rev, on='Product_Code', how='left').fillna(0)

# ── PO features ───────────────────────────────────────────────────────────────
po_df = pd.DataFrame(po_rows)
po_df['Quantity']     = pd.to_numeric(po_df.get('Quantity', 0),     errors='coerce').fillna(0)
po_df['Received_Qty'] = pd.to_numeric(po_df.get('Received_Qty', 0), errors='coerce').fillna(0)

received = po_df.groupby('Prod_Code')['Received_Qty'].sum().reset_index()
received.columns = ['Product_Code', 'total_received']
pending  = po_df[po_df['Received_Qty']==0].groupby('Prod_Code')['Quantity'].sum().reset_index()
pending.columns  = ['Product_Code', 'pending_po_qty']

agg = agg.merge(received, on='Product_Code', how='left').fillna(0)
agg = agg.merge(pending,  on='Product_Code', how='left').fillna(0)
agg['net_stock']           = np.maximum(0, agg['total_received'] - agg['total_qty_sold'])
agg['effective_stock']     = agg['net_stock'] + agg['pending_po_qty']
agg['stock_coverage_days'] = np.where(agg['avg_daily_qty_norm'] > 0, agg['effective_stock'] / agg['avg_daily_qty_norm'], 999)

# ── Wastage ───────────────────────────────────────────────────────────────────
w_df = pd.DataFrame(wastage_rows)
if not w_df.empty and 'Prod_Code' in w_df.columns:
    w_df['Adj_Qty'] = pd.to_numeric(w_df['Adj_Qty'], errors='coerce').fillna(0)
    w_agg = w_df.groupby('Prod_Code')['Adj_Qty'].apply(lambda x: x.abs().sum()).reset_index()
    w_agg.columns = ['Product_Code', 'wastage_qty']
    agg = agg.merge(w_agg, on='Product_Code', how='left').fillna(0)
else:
    agg['wastage_qty'] = 0

# ── Item lookup join ──────────────────────────────────────────────────────────
agg['product_name'] = agg['Product_Code'].map(lambda c: item_lookup.get(str(c),{}).get('name','UNKNOWN PRODUCT'))
agg['cost_price']   = agg['Product_Code'].map(lambda c: item_lookup.get(str(c),{}).get('cost_price', 0))
agg = agg.rename(columns={'Product_Code': 'product_code'})

print(f'\n✅ Feature matrix: {agg.shape[0]} products × {agg.shape[1]} features')
agg[['product_code','product_name','avg_daily_qty_norm','cv_demand','net_stock','effective_stock','stock_coverage_days']].head()


## Step 4 — K-Means SKU Segmentation 🔵

In [ ]:
CLUSTER_FEATURES = ['avg_daily_qty_norm','cv_demand','days_with_sales','total_qty_sold','total_revenue']
CLUSTER_LABELS   = {0:'Fast Mover', 1:'Medium Mover', 2:'Slow Mover', 3:'Dead Stock'}

X_cl = agg[CLUSTER_FEATURES].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cl)

# Elbow method
inertias = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(range(2,9), inertias, 'o-', color=ACCENT, lw=2)
ax.axvline(N_CLUSTERS, color=GREEN, ls='--', label=f'Chosen k={N_CLUSTERS}')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Inertia')
ax.set_title('Elbow Curve — K-Means')
ax.legend()
plt.tight_layout()
plt.show()

# Final fit
km = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
agg['cluster_id'] = km.fit_predict(X_scaled)

centroids = agg.groupby('cluster_id')['avg_daily_qty_norm'].mean().sort_values(ascending=False)
label_map = {cid: CLUSTER_LABELS[rank] for rank, cid in enumerate(centroids.index)}
agg['demand_segment'] = agg['cluster_id'].map(label_map)

# Plot segments
seg_counts = agg['demand_segment'].value_counts()
colors = [ACCENT, BLUE, PURPLE, RED]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(seg_counts.index, seg_counts.values, color=colors[:len(seg_counts)])
ax1.set_title('SKU Count by Demand Segment')
ax1.set_ylabel('Number of SKUs')

ax2.scatter(
    agg['avg_daily_qty_norm'], agg['cv_demand'],
    c=agg['cluster_id'], cmap='tab10', alpha=0.6, s=15
)
ax2.set_xlabel('Avg Daily Qty')
ax2.set_ylabel('CoV (Demand Volatility)')
ax2.set_title('SKU Scatter by Velocity & Volatility')
plt.tight_layout()
plt.show()

for seg, cnt in seg_counts.items():
    print(f'  {seg:15s}: {cnt:,} SKUs')


## Step 5 — XGBoost Regressor (Suggested Qty) 🟠

In [ ]:
XGB_FEATURES = ['avg_daily_qty_norm','cv_demand','days_with_sales','effective_stock','pending_po_qty',
                 'cost_price','wastage_qty','stock_coverage_days']

# Generate training labels using business formula
optimal = np.ceil(agg['avg_daily_qty_norm'] * COVERAGE_DAYS)
agg['target_qty'] = np.maximum(0, np.ceil((optimal - agg['effective_stock']) * (1 + SAFETY_BUFFER))).astype(int)

reorder_mask = (
    (agg['avg_daily_qty_norm'] > 0) &
    (agg['effective_stock'] <= np.ceil(agg['avg_daily_qty_norm'] * LEAD_TIME_DAYS))
)
train_df = agg[reorder_mask].copy()
print(f'Training on {len(train_df):,} products flagged for reorder...')

X = train_df[XGB_FEATURES].fillna(0)
y = train_df['target_qty']

if len(train_df) >= 20:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
else:
    X_train, X_test, y_train, y_test = X, X, y, y

xgb_model = xgb.XGBRegressor(
    n_estimators=200, max_depth=5, learning_rate=0.05,
    objective='reg:squarederror', random_state=42, verbosity=0
)
xgb_model.fit(X_train, y_train)

preds = xgb_model.predict(X_test)
mae   = mean_absolute_error(y_test, preds)
print(f'✅ XGBoost MAE: {mae:.2f} units')

# Feature importance chart
fi = pd.Series(xgb_model.feature_importances_, index=XGB_FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(fi.index, fi.values, color=ACCENT)
ax.set_title('XGBoost — Feature Importance')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()


## Step 6 — Random Forest Urgency Classifier 🟢

In [ ]:
RF_FEATURES = XGB_FEATURES + ['cluster_id']

def urgency_label(cov):
    if cov < 3:  return 'CRITICAL'
    if cov < 7:  return 'HIGH'
    if cov < 14: return 'MEDIUM'
    return 'LOW'

agg['urgency_tier'] = agg['stock_coverage_days'].apply(urgency_label)

tier_counts = agg['urgency_tier'].value_counts()
print('Urgency distribution:')
for t, c in tier_counts.items(): print(f'  {t:8s}: {c:,}')

X_rf = agg[RF_FEATURES].fillna(0)
y_rf = agg['urgency_tier']

if len(agg) >= 20:
    X_tr, X_te, y_tr, y_te = train_test_split(X_rf, y_rf, test_size=0.2, random_state=42, stratify=y_rf)
else:
    X_tr, X_te, y_tr, y_te = X_rf, X_rf, y_rf, y_rf

rf = RandomForestClassifier(n_estimators=150, max_depth=6, random_state=42, class_weight='balanced')
rf.fit(X_tr, y_tr)

rf_preds = rf.predict(X_te)
print('\n', classification_report(y_te, rf_preds, zero_division=0))

# Urgency pie
colors_u = {'CRITICAL': RED, 'HIGH': ACCENT, 'MEDIUM': BLUE, 'LOW': GREEN}
fig, ax = plt.subplots(figsize=(6, 4))
ax.pie(tier_counts.values, labels=tier_counts.index,
       colors=[colors_u.get(t, PURPLE) for t in tier_counts.index],
       autopct='%1.1f%%', startangle=90, textprops={'color':'#f1f5f9'})
ax.set_title('SKU Urgency Distribution')
plt.tight_layout()
plt.show()


## Step 7 — Prophet Demand Forecasting 🔮

> SKUs with fewer than 14 days of sales data use linear extrapolation (avg velocity × 30 days).

In [ ]:
try:
    from prophet import Prophet
    PROPHET_OK = True
except:
    PROPHET_OK = False
    print('⚠️  Prophet not available — using linear fallback for all SKUs.')

forecasts = {}   # product_code → 30-day demand
n_prophet = 0

for code in agg['product_code']:
    sku_daily = daily[daily['Product_Code'] == code]
    fallback  = float(agg.loc[agg['product_code']==code, 'avg_daily_qty_norm'].values[0]) * 30

    if PROPHET_OK and sku_daily['date_parsed'].nunique() >= MIN_PROPHET_DAYS:
        try:
            m = Prophet(weekly_seasonality=True, yearly_seasonality=False, daily_seasonality=False, changepoint_prior_scale=0.3)
            m.fit(sku_daily.rename(columns={'date_parsed':'ds','Quantity':'y'})[['ds','y']])
            future = m.make_future_dataframe(periods=30)
            fc = m.predict(future)
            forecasts[code] = max(0.0, float(fc.tail(30)['yhat'].clip(lower=0).sum()))
            n_prophet += 1
        except:
            forecasts[code] = fallback
    else:
        forecasts[code] = fallback

print(f'✅ Forecasting done: {n_prophet} SKUs used Prophet, {len(agg)-n_prophet} used linear fallback.')


## Step 8 — Generate Predictions & Upload to Supabase 📤

In [ ]:
# ── Apply all model predictions ───────────────────────────────────────────────
agg['forecast_30d']       = agg['product_code'].map(forecasts)
agg['forecast_velocity']  = agg['forecast_30d'] / 30.0

X_all = agg[XGB_FEATURES].fillna(0)
agg['ml_suggested_qty']   = np.maximum(0, np.round(xgb_model.predict(X_all)).astype(int))
agg['ml_urgency']         = rf.predict(agg[RF_FEATURES].fillna(0))
agg['reorder_point']      = np.ceil(agg['avg_daily_qty_norm'] * LEAD_TIME_DAYS).astype(int)
agg['optimal_level']      = np.ceil(agg['avg_daily_qty_norm'] * COVERAGE_DAYS).astype(int)
agg['est_order_value']    = (agg['ml_suggested_qty'] * agg['cost_price']).round(2)

# Print summary
needs_reorder = agg[agg['ml_suggested_qty'] > 0]
print(f'📊 PREDICTION SUMMARY')
print(f'  Total SKUs:          {len(agg):,}')
print(f'  Needing reorder:     {len(needs_reorder):,}')
print(f'  Total units to order:{int(agg["ml_suggested_qty"].sum()):,}')
print(f'  Est. value (KWD):    {agg["est_order_value"].sum():,.2f}')

print('\nTop 10 Most Urgent:')
top10 = needs_reorder.sort_values('stock_coverage_days').head(10)
for _, r in top10.iterrows():
    cov = r['stock_coverage_days']
    print(f'  [{r["product_code"]:10s}] {str(r["product_name"])[:30]:30s} | {r["ml_urgency"]:8s} | suggest: {int(r["ml_suggested_qty"]):5d}')


In [ ]:
# ── Build records ─────────────────────────────────────────────────────────────
now = datetime.now(timezone.utc).isoformat()
records = []
for idx, (_, row) in enumerate(needs_reorder.sort_values('stock_coverage_days').iterrows()):
    code = row['product_code']
    vel  = float(row['avg_daily_qty_norm'])
    f30  = float(forecasts.get(code, vel*30))
    cov  = row['stock_coverage_days']
    reasoning = (
        f"[ML PREDICTION] SKU classified as {row.get('demand_segment','?')} (K-Means). "
        f"Urgency: {row['ml_urgency']} — stock coverage: {'%.1f' % cov if cov<999 else '∞'} days. "
        f"Daily velocity: {vel:.2f} units/day | 30-day forecast: {f30:.1f} units. "
        f"Effective stock: {int(row['effective_stock'])} vs reorder point: {int(row['reorder_point'])}. "
        f"XGBoost recommends ordering {int(row['ml_suggested_qty'])} units "
        f"to reach optimal level of {int(row['optimal_level'])} units + 10% safety buffer."
    )
    records.append({
        'order_id':                  f'ML-2026-{str(idx+1).zfill(4)}',
        'generated_at':              now,
        'product_code':              str(row['product_code']),
        'product_name':              str(row.get('product_name','UNKNOWN')),
        'current_stock_level':       int(row.get('net_stock', 0)),
        'reorder_point':             int(row.get('reorder_point', 0)),
        'optimal_level':             int(row.get('optimal_level', 0)),
        'pending_po_qty':            int(row.get('pending_po_qty', 0)),
        'effective_stock':           int(row.get('effective_stock', 0)),
        'suggested_qty':             int(row.get('ml_suggested_qty', 0)),
        'estimated_order_value_kwd': float(row.get('est_order_value', 0)),
        'daily_sales_velocity':      round(vel, 4),
        'forecast_7day_demand':      round(vel * 7, 1),
        'status':                    'PENDING_APPROVAL',
        'reasoning':                 reasoning,
    })

print(f'📝 {len(records):,} records ready for upload.')


In [ ]:
# ── Clear old records ─────────────────────────────────────────────────────────
clear_url = f'{SUPABASE_URL}/rest/v1/replenishment_orders?order_id=neq.NONEXISTENT'
del_resp  = requests.delete(clear_url, headers={**HEADERS, 'Prefer':'return=minimal'})
print(f'🗑️  Clear existing records: HTTP {del_resp.status_code}')

# ── Batch insert ──────────────────────────────────────────────────────────────
BATCH = 200
inserted = 0
ins_url  = f'{SUPABASE_URL}/rest/v1/replenishment_orders'

for i in range(0, len(records), BATCH):
    batch = records[i:i+BATCH]
    resp  = requests.post(ins_url, headers={**HEADERS, 'Prefer':'return=minimal'}, json=batch)
    if resp.status_code in (200, 201):
        inserted += len(batch)
        print(f'  Batch {i//BATCH+1}: ✅ {len(batch)} rows')
    else:
        print(f'  Batch {i//BATCH+1}: ❌ {resp.status_code} — {resp.text[:200]}')

print(f'\n🎉 Upload complete — {inserted:,} ML predictions in Supabase!')
print('   Open the dashboard to see your ML-powered procurement recommendations.')


## Step 9 — Dashboard Preview 📊

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('☀️ Sunrise Procurement — ML Insights', fontsize=14, color='#f59e0b', y=1.02)

# 1. Segment distribution
seg_counts = agg['demand_segment'].value_counts()
axes[0,0].bar(seg_counts.index, seg_counts.values, color=[ACCENT, BLUE, PURPLE, RED][:len(seg_counts)])
axes[0,0].set_title('SKU Demand Segments')
axes[0,0].set_ylabel('SKU Count')

# 2. Urgency distribution
urg_counts = agg['ml_urgency'].value_counts()
c_map = {'CRITICAL': RED, 'HIGH': ACCENT, 'MEDIUM': BLUE, 'LOW': GREEN}
axes[0,1].bar(urg_counts.index, urg_counts.values, color=[c_map.get(t, PURPLE) for t in urg_counts.index])
axes[0,1].set_title('Predicted Urgency Tiers')
axes[0,1].set_ylabel('SKU Count')

# 3. Suggested qty distribution
axes[1,0].hist(needs_reorder['ml_suggested_qty'].clip(upper=500), bins=40, color=BLUE, alpha=0.8)
axes[1,0].set_title('Suggested Qty Distribution')
axes[1,0].set_xlabel('Units')
axes[1,0].set_ylabel('SKU Count')

# 4. Stock coverage days
cov_clip = agg[agg['stock_coverage_days'] < 50]['stock_coverage_days']
axes[1,1].hist(cov_clip, bins=30, color=GREEN, alpha=0.8)
axes[1,1].axvline(LEAD_TIME_DAYS, color=RED, ls='--', label=f'Lead time ({LEAD_TIME_DAYS}d)')
axes[1,1].set_title('Stock Coverage Distribution (<50 days shown)')
axes[1,1].set_xlabel('Days of Stock')
axes[1,1].legend()

plt.tight_layout()
plt.show()
print('\n✅ All done! Open http://localhost:3456 to view the live dashboard.')
